In [1]:
import numpy as np
import pandas as pd
#import scipy.stats as sps
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import tt_ind_solve_power

**CUPED** (Controlled-experiment Using Pre-Experiment Data) — это статистический метод увеличения чувствительности A/B-тестов за счет использования исторических данных пользователей, собранных до начала эксперимента.  
Говоря простыми словами, метод позволяет ускорить проведение A/B-тестов и быстрее обнаружить эффект от изменений в продукте, не увеличивая при этом объем привлекаемого трафика.  
Это обход проблемы недостаточной чувствительности теста.

**Задача:**  
- Был проведен тест новой системы цен на очень небольшую аудиторию. Профессионалы в небольшом городе.
- Аудитория пассивная, поэтому в ней редко "прокрашиваются" фичи. Каждая потенциальная фича на вес золота.
- Менеджеры имеют большие надежды на эту фичу.
- Основная метрика ARPPU.

## Функции для использования метода CUPED

In [7]:
def generate_corr_data(N, pre_mean, post_mean, corr = 0.8):
    """Генерирация датафрейма с нужной корреляцией между 2 ковариатами.

    N - размер data frame
    pre_mean - средняя до старта теста
    post_mean - средняя после старта теста
    corr - корреляция метрики во время эксперимента с метрикой до эксперимента.
    Возвращает датафрейм с коррелированной метрикой до эксперимента и после эксперимента
    """
    cov = [[1, corr], [corr, 1]]
    correlated_data = np.random.multivariate_normal([pre_mean, post_mean], cov, N)
    correlated_df = pd.DataFrame({'post_value': correlated_data[:, 1], 'pre_value': correlated_data[:, 0]})
    return (correlated_df)

def calculate_theta_basic(pre, post):
    theta = (np.cov(post, pre)[0, 1]) / (np.var(pre))
    return(theta)

def calculate_theta_for_test(control_pre, control_post, test_pre, test_post):
    theta = (np.cov(control_post, control_pre)[0, 1] + np.cov(test_post, test_pre)[0, 1]) /\
            (np.var(control_pre) + np.var(test_pre))
    return(theta)

In [8]:
def get_basic_ttest(group_A, group_B):
    '''Проверяет гипотезу о равенстве средних для обычного среднего.
    return - t_stat, p_value'''

    t_stat, p_value = stats.ttest_ind(group_A, group_B)
    inference = {'t_stat': t_stat, 'p_value':p_value}
    return(inference)

def get_cuped_ttest(control_pre, control_post, test_pre, test_post):
    '''Проверяет гипотезу о равенстве средних CUPED вариант
    return - t_stat, p_value'''

    theta = calculate_theta_for_test(control_pre, control_post, test_pre, test_post)

    control_cuped = control_post - theta * control_pre
    test_cuped = test_post - theta * test_pre

    inference = get_basic_ttest(control_cuped, test_cuped)

    return(inference)

# Загружаем данные

In [14]:
# загружаем данные по тесту
data = pd.read_csv('E:\Projects\portfolio\causal_inference_method\data\cuped.csv')
data.head()

,post_ARPPU,pre_ARPPU,group
0,660.0,595.0,A
1,540.0,621.0,A
2,863.0,782.0,A
3,431.0,567.0,A
4,434.0,473.0,A


## Первичный анализ
- Наблюдаемая абсольтная разница между средними у Б и А групп.
- Наблюдаемый effect size Cohen D. Для рассчетов используется асолютнная разница и стандартное отклонение в pre данных в группе А

In [15]:
# Средние по группам
agg_df = data.groupby('group').mean()
agg_df

,post_ARPPU,pre_ARPPU
group,,
A,598.689905,596.972149
B,604.045757,597.265784


In [16]:
abs_observed_lift = agg_df['post_ARPPU']['B'] - agg_df['post_ARPPU']['A']
print('Абсолютный наблюдаемый эффект =', abs_observed_lift)

cohen_d = abs_observed_lift / data['pre_ARPPU'][data['group']=='A'].std()
print('Наблюдаемый Cohen D размер эффекта =', cohen_d)

Абсолютный наблюдаемый эффект = 5.355852506777865
Наблюдаемый Cohen D размер эффекта = 0.05415716452599449


## Простой t-test для POST данных

In [17]:
# p-value полученное при проведении простого t-test на post данных
# проведение простого t-test. Получение p_value
inference =  get_basic_ttest(data['post_ARPPU'][data['group'] == 'A'], data['post_ARPPU'][data['group'] == 'B'])
print('p-value простого t-test =', round(inference['p_value'], 3))

p-value простого t-test = 0.074


- t-test не показал стат. значимое различие.
- Проба использования pre данных. Возможно они помогут повысить чувствительность теста.

## Проведение Cuped t-test
- Доля пустых значений в pre данных
- p-value в проведенном cuped t-test

In [18]:
# доля пустых значений в pre данных
print('Доля пустых значений в pre данных равна =', round(data['pre_ARPPU'].isna().sum()/ len(data['pre_ARPPU']), 3))

Доля пустых значений в pre данных равна = 0.133


In [ ]:
# заполнение пустых значений средней на pre периоде
pre_mean = data['pre_ARPPU'].mean()
data['pre_ARPPU'] = data['pre_ARPPU'].fillna(pre_mean)


In [20]:
# разобиение на соответствующие группы для того чтобы провести cuped t-test
pre_control = data['pre_ARPPU'][data['group'] == 'A']
post_control = data['post_ARPPU'][data['group'] == 'A']
pre_test = data['pre_ARPPU'][data['group'] == 'B']
post_test = data['post_ARPPU'][data['group'] == 'B']

In [21]:
# Проведение CUPED t-test
basic_ttest_inference = get_basic_ttest(data['post_ARPPU'][data['group'] == 'A'], data['post_ARPPU'][data['group'] == 'B'])
cuped_ttest_inference = get_cuped_ttest(pre_control, post_control, pre_test, post_test)

In [22]:
# p-value в проведенном cuped t-test
print('p-value в cuped t-test', round(cuped_ttest_inference['p_value'], 3))

p-value в cuped t-test 0.032


# Рассчет размера обычной и CUPED выборки
- альфа = 0.05 и мощности 0.8
- двусторонняя проверка гипотезы
- размер выборок 50/50
- статистический критерий t-test
- относительный MDE = 0.01
- за основу взят другой датасет на исторических данных cuped_pre_pre_data.csv. Он имеет похожую схему. Только отсутствует деление на группы.

In [ ]:
# загружаем данные
# pre_pre_data = pd.read_csv('cuped_pre_pre_data.csv')
pre_pre_data = data[['post_ARPPU','pre_ARPPU']].copy()
pre_pre_data.head()

,post_ARPPU,pre_ARPPU
0,660.0,595.0
1,540.0,621.0
2,863.0,782.0
3,431.0,567.0
4,434.0,473.0


In [ ]:
# рассчитаем размер выборки нужный для базового теста

# заполним пустые значения средней на pre периоде
pre_mean = pre_pre_data['pre_ARPPU'].mean()
pre_pre_data['pre_ARPPU'] = pre_pre_data['pre_ARPPU'].fillna(pre_mean)

# рассчитаем контрольные средние и std. Берем post период для обычной выборки
control_std = pre_pre_data['post_ARPPU'].std() # в функции generate_corr_data генерируются нормальные величины с std = 1
control_mean = pre_pre_data['post_ARPPU'].mean()
rel_effect = 0.01
mean_diff = control_mean * rel_effect
alpha = 0.05
power = 0.8


# Расчет индекса Коэна
cohen_d_basic  = mean_diff / control_std
print(cohen_d_basic)
n = tt_ind_solve_power(effect_size = cohen_d_basic,
                       alpha = alpha,
                       power = power,
                       ratio = 1,
                       alternative = "two-sided")
basic_sample_size = round(n)
print('Basic sample size =', basic_sample_size)

theta = calculate_theta_basic(pre_pre_data['pre_ARPPU'], pre_pre_data['post_ARPPU'])
Y_cuped = pre_pre_data['post_ARPPU'] - theta * pre_pre_data['pre_ARPPU']
Y_cuped_std = Y_cuped.std()

cohen_d_cuped  = mean_diff / Y_cuped_std
print(cohen_d_cuped)
n_cuped = tt_ind_solve_power(effect_size = cohen_d_cuped,
                       alpha = alpha,
                       power = 0.8,
                       ratio = 1,
                       alternative = "two-sided")
cuped_sample_size = round(n_cuped)
print('CUPED sample size', cuped_sample_size)

0.05999864724775356
Basic sample size = 4362
0.074481718000916
CUPED sample size 2831


In [40]:
# Значение theta по рассчетам
print('Theta равна = ', theta)

Theta равна =  0.6291182153058155


In [41]:
# Стандартное отклонение Y_cuped мотносительно стандартного отклоения post_ARPPU
rel_diff_in_std = abs(Y_cuped_std / control_std - 1)
print('Cтандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU на {} %'.format(round(rel_diff_in_std*100, 1)))

Cтандартное отклонение Y_cuped меньше стандартного отклоения post_ARPPU на 19.4 %


In [ ]:
# Требуемый размер выборки нужный для детекции MDE на post данных простым t-test
print('Размер выборки для обычного t-test =', basic_sample_size)

Размер выборки для обычного t-test = 4362


In [43]:
# Размер выборки нужный для детекции MDE CUPED t-test
print('Размер выборки для CUPED t-test =', cuped_sample_size)

Размер выборки для CUPED t-test = 2831
